<a href="https://colab.research.google.com/github/heri-espino/LDS1071-Machine-Learning/blob/main/notebooks/14_Text-classification_TFIDF-vs-CountVectorizer_R3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica 7 · Clasificación de textos
### TF-IDF vs. CountVectorizer + Regresión Logística

**Objetivo:** comparar cómo cambia la clasificación cuando representamos los mismos documentos con TF-IDF o con conteos de palabras.

Basado en **Clase 20**, diapositivas 29–44. Se usan las cuatro categorías originales de *20 Newsgroups*. Las explicaciones están en español y los ejemplos de texto en inglés.

## 1. Librerías y datos

Entrenaremos dos clasificadores con **exactamente los mismos documentos**. Usaremos las etiquetas originales solo para aprendizaje supervisado y evaluación.

> Ejecutar en Google Colab o Jupyter con Internet: la primera carga del dataset puede descargar archivos.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook", palette="tab10")

from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

categorias = [
    "sci.space", "rec.sport.baseball", "comp.graphics", "talk.politics.mideast"
]

datos = fetch_20newsgroups(
    subset="train", categories=categorias,
    remove=("headers", "footers", "quotes"),
    shuffle=True, random_state=42
)
documentos = datos.data
etiquetas = datos.target
nombres = datos.target_names

print("Documentos:", len(documentos))
print("Categorías:", nombres)

Documentos: 2338
Categorías: ['comp.graphics', 'rec.sport.baseball', 'sci.space', 'talk.politics.mideast']


In [2]:
i = 11
print("Categoría:", nombres[etiquetas[i]])
print("Texto:\n", documentos[i][:500])

Categoría: rec.sport.baseball
Texto:
 




I'm not sure I understand this question. When the IF rule is invoked,
the batter is automatically out. This relieves the runners from being
forced to advance to the next base if the ball is not caught. Other
than that, isn't it just the same as any situation in which a runner on
a base is not forced to the next base on a dropped fly ball? That is,
if the ball is caught he can tag up and run (or decide to stay), and
if the ball is dropped he can have left the base at any time.



## 2. Entrenamiento y prueba

Usamos **80 %** de los documentos para entrenar y **20 %** para evaluar. `stratify` conserva aproximadamente la proporción de cada categoría.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    documentos, etiquetas,
    test_size=0.2, random_state=42, stratify=etiquetas
)
print("Entrenamiento:", len(X_train))
print("Prueba:", len(X_test))

Entrenamiento: 1870
Prueba: 468


## 3. Dos pipelines: TF-IDF y CountVectorizer

- **TF-IDF:** pondera palabras según frecuencia y rareza; además normaliza con L2.
- **CountVectorizer:** cuenta apariciones. **No** utiliza `norm="l2"`, pues ese parámetro no existe en `CountVectorizer`.

El clasificador y sus hiperparámetros se mantienen iguales para que la comparación se concentre en la representación.

In [4]:
parametros = dict(
    stop_words="english",
    min_df=5,
    max_df=0.8,
    max_features=5000
)

modelo_tfidf = Pipeline([
    ("vectorizador", TfidfVectorizer(**parametros, norm="l2")),
    ("clasificador", LogisticRegression(max_iter=1000, random_state=42))
])

modelo_count = Pipeline([
    ("vectorizador", CountVectorizer(**parametros)),
    ("clasificador", LogisticRegression(max_iter=1000, random_state=42))
])

modelos = {"TF-IDF": modelo_tfidf, "CountVectorizer": modelo_count}
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    print(nombre, "→ vocabulario:", len(modelo.named_steps["vectorizador"].get_feature_names_out()))

TF-IDF → vocabulario: 5000
CountVectorizer → vocabulario: 5000


## 4. Evaluación en los mismos documentos

- **Accuracy:** proporción de aciertos.
- **Precision:** aciertos entre los predichos en una clase.
- **Recall:** aciertos entre los verdaderos de una clase.
- **F1:** media armónica de precision y recall.

In [5]:
predicciones = {}
resultados = []

for nombre, modelo in modelos.items():
    pred = modelo.predict(X_test)
    predicciones[nombre] = pred
    acc = accuracy_score(y_test, pred)
    resultados.append({"Representación": nombre, "Accuracy": acc})
    print(f"\n{nombre} — Accuracy: {acc:.3f}")
    print(classification_report(y_test, pred, target_names=nombres))

tabla_resultados = pd.DataFrame(resultados)
display(tabla_resultados.style.format({"Accuracy": "{:.3f}"}))


TF-IDF — Accuracy: 0.868
                       precision    recall  f1-score   support

        comp.graphics       0.90      0.85      0.88       117
   rec.sport.baseball       0.78      0.94      0.85       119
            sci.space       0.87      0.82      0.84       119
talk.politics.mideast       0.94      0.86      0.90       113

             accuracy                           0.87       468
            macro avg       0.87      0.87      0.87       468
         weighted avg       0.87      0.87      0.87       468


CountVectorizer — Accuracy: 0.823
                       precision    recall  f1-score   support

        comp.graphics       0.89      0.79      0.84       117
   rec.sport.baseball       0.72      0.93      0.81       119
            sci.space       0.81      0.78      0.79       119
talk.politics.mideast       0.94      0.78      0.85       113

             accuracy                           0.82       468
            macro avg       0.84      0.82      0.82

,Representación,Accuracy
0,TF-IDF,0.868
1,CountVectorizer,0.823


### Interpretación de la evaluación

En **468 documentos de prueba**, TF-IDF clasificó correctamente **406 (accuracy = 0.868)**, frente a **385 (accuracy = 0.823)** con CountVectorizer: **21 aciertos más**, una diferencia de **4.5 puntos porcentuales**.

**TF-IDF obtuvo un F1 mayor en las cuatro categorías:** gráficos (0.88 frente a 0.84), béisbol (0.85 frente a 0.81), espacio (0.84 frente a 0.79) y política de Medio Oriente (0.90 frente a 0.85). La comparación utiliza exactamente la misma división de entrenamiento y prueba.


In [ ]:
# Gráfico comparativo con estilo Seaborn whitegrid.
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=tabla_resultados, x="Representación", y="Accuracy",
            hue="Representación", palette="tab10", legend=False, ax=ax)
ax.set(ylim=(0, 1), ylabel="Accuracy", xlabel="",
       title="Exactitud en el conjunto de prueba")
for i, valor in enumerate(tabla_resultados["Accuracy"]):
    ax.text(i, valor + 0.02, f"{valor:.3f}", ha="center")
fig.tight_layout()
plt.show()

### Matrices de confusión

La **diagonal** indica predicciones correctas. Fuera de ella están los errores.

In [ ]:
# Dos matrices de confusión en formato heatmap.
from sklearn.metrics import confusion_matrix

fig, ejes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (nombre, pred) in zip(ejes, predicciones.items()):
    matriz = confusion_matrix(y_test, pred, labels=np.arange(len(nombres)))
    sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=nombres, yticklabels=nombres, ax=ax)
    ax.set(title=nombre, xlabel="Predicción", ylabel="Categoría real")
    ax.tick_params(axis="x", rotation=35)
    ax.tick_params(axis="y", rotation=0)
fig.tight_layout()
plt.show()

### Interpretación de las matrices de confusión

La diagonal muestra los aciertos y las celdas restantes, las confusiones. **Béisbol tiene el mayor recall** (0.94 en TF-IDF, 0.93 en CountVectorizer), pero una precisión más baja (0.78 y 0.72): se recuperan muchos textos de béisbol, aunque otros documentos también reciben esa etiqueta. El menor rendimiento de CountVectorizer aparece en todas las categorías, no solo en una.


## 5. Examinar aciertos y errores

Mostramos ejemplos de ambos modelos. Los errores ayudan a detectar textos ambiguos, demasiado cortos o con vocabulario compartido.

In [8]:
def mostrar_ejemplos(nombre, aciertos=True, cantidad=3):
    pred = predicciones[nombre]
    indices = np.where(pred == y_test if aciertos else pred != y_test)[0]
    print(f"{nombre}: {len(indices)} {'aciertos' if aciertos else 'errores'}")
    for i in indices[:cantidad]:
        print("\n---")
        print("Real:", nombres[y_test[i]])
        print("Predicción:", nombres[pred[i]])
        print(X_test[i][:350].replace("\n", " "))

for nombre in modelos:
    mostrar_ejemplos(nombre, aciertos=True, cantidad=2)
    mostrar_ejemplos(nombre, aciertos=False, cantidad=2)

TF-IDF: 406 aciertos

---
Real: sci.space
Predicción: sci.space
 A cash award is OK. A time limit would be nice. You can't give away mining rights (assuming there's anything to mine) because you don't own them.  -----------------------------------------------------------------  .sig files are like strings ... every yo-yo's got one.

---
Real: comp.graphics
Predicción: comp.graphics
Howdy all,  	I was wondering if people could e-mail me their opinions on the various graphics viewers available for MS-Windows 3.x...  I'm working on a project to set up our scanner and write documentation on how to use it and it would be nice to have a snazzy image viewer  to look at (and maybe even edit?) the image you just scanned.  The file for
TF-IDF: 62 errores

---
Real: talk.politics.mideast
Predicción: rec.sport.baseball
You a good case for rights to abortion.

---
Real: sci.space
Predicción: talk.politics.mideast
Sounds similar to something Wally Schirra said.
CountVectorizer: 385 aciertos

---
Rea

### Interpretación de los ejemplos

TF-IDF cometió **62 errores** y CountVectorizer **83**. Ambos clasifican incorrectamente ejemplos muy breves, como la frase sobre *abortion* asignada a béisbol, o la referencia a *Wally Schirra* asignada a política en lugar de espacio. Los textos cortos o ambiguos pueden contener pocas señales temáticas útiles; estos ejemplos ilustran errores particulares, no explican todos los fallos.


## 6. Coeficientes: palabras más influyentes

En una clasificación con cuatro categorías, cada clase tiene su propia fila de coeficientes. Un peso **positivo** favorece esa categoría frente a las demás; uno **negativo** la desfavorece. Comparamos los diez extremos de cada fila.

In [9]:
def palabras_influyentes(modelo, cantidad=10):
    terminos = modelo.named_steps["vectorizador"].get_feature_names_out()
    clf = modelo.named_steps["clasificador"]
    filas = []
    for fila, clase in enumerate(clf.classes_):
        pesos = clf.coef_[fila]
        positivos = np.argsort(pesos)[-cantidad:][::-1]
        negativos = np.argsort(pesos)[:cantidad]
        filas.append({
            "Categoría": nombres[clase],
            "Positivas (+)": ", ".join(terminos[positivos]),
            "Negativas (−)": ", ".join(terminos[negativos])
        })
    return pd.DataFrame(filas)

for nombre, modelo in modelos.items():
    print("\n", nombre)
    display(palabras_influyentes(modelo))


 TF-IDF


,Categoría,Positivas (+),Negativas (−)
0,comp.graphics,"graphics, image, file, hi, computer, 3d, files...","space, people, year, israel, baseball, jewish,..."
1,rec.sport.baseball,"baseball, game, year, team, games, players, st...","space, israel, graphics, software, israeli, pr..."
2,sci.space,"space, nasa, orbit, launch, moon, shuttle, ear...","graphics, israel, jewish, israeli, file, baseb..."
3,talk.politics.mideast,"israel, israeli, jews, people, turkish, armeni...","space, thanks, year, graphics, baseball, nasa,..."



 CountVectorizer


,Categoría,Positivas (+),Negativas (−)
0,comp.graphics,"graphics, hi, image, file, looking, ve, comput...","people, space, appreciate, idea, moon, jewish,..."
1,rec.sport.baseball,"baseball, stadium, team, writes, game, games, ...","space, try, israel, real, loser, message, grap..."
2,sci.space,"space, orbit, launch, nasa, moon, things, appr...","graphics, jewish, point, ve, israel, agree, co..."
3,talk.politics.mideast,"israel, loser, israeli, jews, serdar, turkish,...","got, thanks, space, lot, good, actually, looki..."


### Interpretación de los coeficientes

Los dos modelos identifican pistas temáticas similares: *graphics, image, file* para gráficos; *baseball, game, team* para béisbol; *space, nasa, orbit* para espacio; e *israel, israeli, jews* para política de Medio Oriente. Algunas palabras comunes o nombres propios también reciben pesos elevados.

Un coeficiente positivo favorece una clase **en relación con las otras clases**, y uno negativo la desfavorece. **No conviene comparar directamente la magnitud de coeficientes entre TF-IDF y conteos**, porque cambia la escala de entrada.


In [ ]:
# Coeficientes con Seaborn. Solo la primera clase para evitar saturación.
fig, ejes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (nombre, modelo) in zip(ejes, modelos.items()):
    terminos = modelo.named_steps["vectorizador"].get_feature_names_out()
    clasificador = modelo.named_steps["clasificador"]
    pesos = clasificador.coef_[0]
    indices = np.argsort(np.abs(pesos))[-12:]
    tabla_pesos = pd.DataFrame({
        "Término": terminos[indices], "Coeficiente": pesos[indices]
    }).sort_values("Coeficiente")
    sns.barplot(data=tabla_pesos, x="Coeficiente", y="Término",
                color=sns.color_palette("tab10")[0], ax=ax)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"{nombre}: {nombres[clasificador.classes_[0]]}")
fig.tight_layout()
plt.show()

## 7. Una noticia real en inglés

**Noticia:** “Brewers sweep the Dodgers in the NLCS to reach the World Series for the first time since 1982”  
**Fuente:** Associated Press (Beth Harris), **15 de octubre de 2026**.  
**Enlace:** https://apnews.com/article/brewers-dodgers-score-nlcs-cfd30befcec42a035cbea5cae3bf746c

Para respetar los derechos de autor, el texto siguiente es un **resumen original en inglés de la noticia**, no un fragmento copiado de AP. El evento real corresponde a la categoría esperada `rec.sport.baseball`. Puedes sustituir `noticia` por un fragmento breve del artículo para comparar.

In [11]:
titulo_noticia = "Brewers sweep the Dodgers in the NLCS to reach the World Series for the first time since 1982"
url_noticia = "https://apnews.com/article/brewers-dodgers-score-nlcs-cfd30befcec42a035cbea5cae3bf746c"

noticia = """
The Milwaukee Brewers advanced to the World Series after defeating the Los Angeles
Dodgers in four games of the National League Championship Series. In the final
baseball game, Milwaukee won 6-4 after overcoming an early Dodgers lead.
Garrett Mitchell hit a go-ahead home run in the sixth inning, while the Brewers'
bullpen and relief pitchers protected the lead. Shohei Ohtani homered for Los
Angeles, but Milwaukee completed the playoff sweep. The Brewers finished the
regular season with 103 victories and will return to the World Series for the
first time since 1982. Baseball fans celebrated the National League pennant.
""".strip()

print(titulo_noticia)
print(url_noticia)
print("\nTexto analizado (resumen original):\n", noticia)

Brewers sweep the Dodgers in the NLCS to reach the World Series for the first time since 1982
https://apnews.com/article/brewers-dodgers-score-nlcs-cfd30befcec42a035cbea5cae3bf746c

Texto analizado (resumen original):
 The Milwaukee Brewers advanced to the World Series after defeating the Los Angeles
Dodgers in four games of the National League Championship Series. In the final
baseball game, Milwaukee won 6-4 after overcoming an early Dodgers lead.
Garrett Mitchell hit a go-ahead home run in the sixth inning, while the Brewers'
bullpen and relief pitchers protected the lead. Shohei Ohtani homered for Los
Angeles, but Milwaukee completed the playoff sweep. The Brewers finished the
regular season with 103 victories and will return to the World Series for the
first time since 1982. Baseball fans celebrated the National League pennant.


### Clasificación y probabilidades

Ambos modelos ven **exactamente el mismo texto**. `predict_proba` estima la distribución entre las cuatro categorías posibles; son probabilidades del modelo, no garantías de corrección.

In [12]:
filas = []
for nombre, modelo in modelos.items():
    etiqueta = modelo.predict([noticia])[0]
    clf = modelo.named_steps["clasificador"]
    probabilidades = modelo.predict_proba([noticia])[0]
    print(f"{nombre} → categoría predicha: {nombres[etiqueta]}")
    for clase, prob in zip(clf.classes_, probabilidades):
        filas.append({"Representación": nombre, "Categoría": nombres[clase], "Probabilidad": prob})

tabla_noticia = pd.DataFrame(filas)
display(tabla_noticia.pivot(index="Categoría", columns="Representación", values="Probabilidad").round(3))

TF-IDF → categoría predicha: rec.sport.baseball
CountVectorizer → categoría predicha: rec.sport.baseball


Representación,CountVectorizer,TF-IDF
Categoría,,
comp.graphics,0.0,0.030
rec.sport.baseball,1.0,0.899
sci.space,0.0,0.035
talk.politics.mideast,0.0,0.036


### Interpretación de la noticia

Ambos modelos asignan la noticia a **rec.sport.baseball**, su tema esperado. TF-IDF da una probabilidad aproximada de **0.899**, mientras que CountVectorizer produce un valor que **redondea a 1.000**. Una probabilidad más extrema **no garantiza mejor precisión ni calibración**. El resultado del conjunto de prueba sigue favoreciendo a TF-IDF; una sola noticia no sustituye esa evaluación.


In [ ]:
# Probabilidades de clasificación del mismo artículo.
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(data=tabla_noticia, x="Categoría", y="Probabilidad",
            hue="Representación", order=nombres, ax=ax)
ax.set(ylim=(0, 1.05), ylabel="Probabilidad estimada", xlabel="",
       title="Noticia: TF-IDF vs. CountVectorizer")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
plt.show()

## 8. Discusión y conclusión

Observa las métricas, las palabras y las predicciones anteriores. **No compares directamente la magnitud de los coeficientes** entre representaciones: sus escalas de entrada son diferentes.

In [14]:
acc_tfidf = tabla_resultados.loc[tabla_resultados["Representación"] == "TF-IDF", "Accuracy"].iloc[0]
acc_count = tabla_resultados.loc[tabla_resultados["Representación"] == "CountVectorizer", "Accuracy"].iloc[0]

ganador = "TF-IDF" if acc_tfidf > acc_count else "CountVectorizer" if acc_count > acc_tfidf else "Empate"
print(f"Accuracy TF-IDF:        {acc_tfidf:.3f}")
print(f"Accuracy CountVectorizer: {acc_count:.3f}")
print("Mejor accuracy:", ganador)
print("Diferencia absoluta:", f"{abs(acc_tfidf - acc_count):.3f}")

for nombre, modelo in modelos.items():
    etiqueta = modelo.predict([noticia])[0]
    prob = modelo.predict_proba([noticia]).max()
    print(f"Noticia ({nombre}): {nombres[etiqueta]} — {prob:.3f}")

Accuracy TF-IDF:        0.868
Accuracy CountVectorizer: 0.823
Mejor accuracy: TF-IDF
Diferencia absoluta: 0.045
Noticia (TF-IDF): rec.sport.baseball — 0.899
Noticia (CountVectorizer): rec.sport.baseball — 1.000


### Conclusión

**TF-IDF funcionó mejor en esta práctica:** obtuvo mayor accuracy y F1 en todas las clases. Una explicación plausible es que su ponderación reduce el protagonismo de términos frecuentes y que la normalización L2 atenúa la influencia de la longitud de los documentos. CountVectorizer conserva conteos sin normalizar, lo que puede amplificar palabras repetidas. No obstante, los resultados **no permiten aislar** qué parte de la mejora se debe a IDF y cuál a la normalización; para ello se necesitaría un tercer experimento controlado.

Para la noticia de béisbol, ambos predijeron la categoría correcta, pero la confianza extrema de CountVectorizer no significa que generalice mejor.


**Responde brevemente después de ejecutar:**

1. ¿Qué representación obtuvo mayor accuracy y cuál tuvo mejores valores F1 por categoría?
2. ¿Qué confusiones aparecen en las matrices? ¿Coinciden entre modelos?
3. ¿Qué palabras positivas y negativas explican mejor alguna categoría?
4. ¿Qué modelos clasificaron correctamente la noticia de béisbol? Compara sus probabilidades.
5. ¿Por qué los conteos brutos podrían dar más importancia a documentos largos que TF-IDF normalizado?

**Conclusión:** la representación modifica las características de entrada y, por tanto, las decisiones aprendidas por el clasificador. La mejor opción se decide con las métricas del conjunto de prueba, no solamente con una noticia.

# R3 visualization

## 9. De PCA a la esfera y después a la clasificación

Exploraremos cada representación mediante cuatro etapas:

1. **PCA:** proyectar los documentos a tres dimensiones, todavía sin colores.
2. **Esfera:** normalizar las coordenadas 3D con norma L2 para comparar sus direcciones.
3. **Clasificación:** colorear los puntos según las categorías predichas por la regresión logística **ya entrenada en el espacio original**.
4. **Vecinos de la noticia:** destacar documentos cercanos al artículo mediante coseno, comparando la esfera 3D con el espacio original.

También podemos mostrar aciertos/errores y categorías verdaderas **solo para evaluar**.

**Importante:** PCA y la esfera son una herramienta de visualización. Los clasificadores siguen usando TF-IDF o CountVectorizer con sus dimensiones completas; **no entrenamos K-means**, porque esta actividad es de clasificación supervisada.


In [ ]:
# Plotly: rotación interactiva. Seaborn whitegrid se usa en las figuras 2D.
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

# La misma muestra de documentos para ambos métodos.
indices_3d = np.sort(np.random.default_rng(42).choice(
    len(X_test), size=min(450, len(X_test)), replace=False
))
colores = sns.color_palette("tab10", n_colors=len(nombres)).as_hex()
n_vecinos = 15

In [ ]:
# Transformaciones R3: primero PCA y después normalización.
datos_r3 = {}

for nombre, modelo in modelos.items():
    vectorizador = modelo.named_steps["vectorizador"]
    textos_muestra = [X_test[i] for i in indices_3d]
    X_muestra = vectorizador.transform(textos_muestra)
    X_noticia = vectorizador.transform([noticia])

    pca = PCA(n_components=3, svd_solver="randomized", random_state=42)
    coords = pca.fit_transform(X_muestra.toarray().astype(np.float32))
    noticia_3d = pca.transform(X_noticia.toarray().astype(np.float32))

    esfera = normalize(coords, norm="l2")
    noticia_esfera = normalize(noticia_3d, norm="l2")

    # Dos definiciones de vecinos: proyección 3D y representación original.
    similitud_3d = cosine_similarity(esfera, noticia_esfera).ravel()
    similitud_original = cosine_similarity(X_muestra, X_noticia).ravel()
    vecinos_3d = np.argsort(similitud_3d)[-n_vecinos:]
    vecinos_originales = np.argsort(similitud_original)[-n_vecinos:]

    datos_r3[nombre] = {
        "coords": coords, "esfera": esfera,
        "noticia_esfera": noticia_esfera,
        "predichas": predicciones[nombre][indices_3d],
        "reales": np.asarray(y_test)[indices_3d],
        "vecinos_3d": vecinos_3d,
        "vecinos_originales": vecinos_originales,
        "varianza": pca.explained_variance_ratio_.sum(),
    }
    coinciden = len(set(vecinos_3d) & set(vecinos_originales))
    print(f"{nombre}: PCA 3D explica {datos_r3[nombre]['varianza']:.1%} de la varianza.")
    print(f"Vecinos en común (3D vs. original): {coinciden}/{n_vecinos}")

In [ ]:
def grafica_3d(nombre):
    d = datos_r3[nombre]
    coords, esfera = d["coords"], d["esfera"]
    predichas, reales = d["predichas"], d["reales"]
    noticia_esfera = d["noticia_esfera"]

    # Malla transparente de la esfera unitaria.
    phi = np.linspace(0, 2 * np.pi, 36)
    theta = np.linspace(0, np.pi, 22)
    sx = np.outer(np.cos(phi), np.sin(theta))
    sy = np.outer(np.sin(phi), np.sin(theta))
    sz = np.outer(np.ones_like(phi), np.cos(theta))

    fig = go.Figure()
    vistas = {}

    def puntos(x, ids, etiqueta, color, tamano=3, opacidad=0.65, leyenda=False):
        fig.add_trace(go.Scatter3d(
            x=x[ids, 0], y=x[ids, 1], z=x[ids, 2],
            mode="markers", name=etiqueta,
            marker=dict(color=color, size=tamano, opacity=opacidad),
            customdata=indices_3d[ids],
            hovertemplate="Documento de prueba: %{customdata}<extra>%{fullData.name}</extra>",
            showlegend=leyenda, visible=False))
        return len(fig.data) - 1

    def superficie():
        fig.add_trace(go.Surface(
            x=sx, y=sy, z=sz, opacity=0.10,
            colorscale=[[0, "#93c5fd"], [1, "#93c5fd"]],
            showscale=False, hoverinfo="skip", visible=False))
        return len(fig.data) - 1

    def noticia():
        fig.add_trace(go.Scatter3d(
            x=noticia_esfera[:, 0], y=noticia_esfera[:, 1],
            z=noticia_esfera[:, 2], mode="markers",
            marker=dict(size=10, color="#111827", symbol="diamond"),
            name="Noticia", hovertext=titulo_noticia,
            hoverinfo="text", visible=False, showlegend=True))
        return len(fig.data) - 1

    todos = np.arange(len(indices_3d))
    vistas["1. PCA"] = [puntos(coords, todos, "Documentos", "#64748b")]
    vistas["2. Esfera"] = [
        superficie(), puntos(esfera, todos, "Documentos normalizados", "#64748b")]

    def por_clase(etiquetas, prefijo):
        ids = [superficie()]
        for k, categoria in enumerate(nombres):
            miembros = np.flatnonzero(etiquetas == k)
            ids.append(puntos(esfera, miembros, categoria, colores[k], leyenda=True))
        return ids

    vistas["3. Clasificación"] = por_clase(predichas, "Predicción")

    aciertos = np.flatnonzero(reales == predichas)
    errores = np.flatnonzero(reales != predichas)
    vistas["4. Aciertos/errores"] = [
        superficie(),
        puntos(esfera, aciertos, "Aciertos", "#16a34a", leyenda=True),
        puntos(esfera, errores, "Errores", "#dc2626", tamano=5, leyenda=True),
    ]

    for titulo, campo in [
        ("5. Vecinos 3D", "vecinos_3d"),
        ("6. Vecinos originales", "vecinos_originales"),
    ]:
        vistas[titulo] = [
            superficie(),
            puntos(esfera, todos, "Otros documentos", "#94a3b8", opacidad=0.12),
            puntos(esfera, d[campo], "Vecinos de la noticia", "#f59e0b",
                   tamano=6, opacidad=1, leyenda=True),
            noticia(),
        ]

    vistas["7. Categorías reales"] = por_clase(reales, "Real")

    def escena(esfera_visible):
        estilo = dict(showbackground=False, gridcolor="#e2e8f0")
        if esfera_visible:
            estilo["range"] = [-1.12, 1.12]
        return dict(xaxis=dict(title="PC 1", **estilo),
                    yaxis=dict(title="PC 2", **estilo),
                    zaxis=dict(title="PC 3", **estilo),
                    aspectmode="cube" if esfera_visible else "data")

    botones = []
    for vista, ids in vistas.items():
        botones.append(dict(
            label=vista, method="update",
            args=[{"visible": [i in ids for i in range(len(fig.data))]},
                  {"title": f"{nombre} · {vista}",
                   "scene": escena(vista != "1. PCA")}],
        ))
    for i in vistas["1. PCA"]:
        fig.data[i].visible = True

    fig.update_layout(
        template="plotly_white", title=f"{nombre} · 1. PCA",
        height=650, margin=dict(l=0, r=0, b=0, t=120),
        legend=dict(x=0.01, y=0.98, bgcolor="rgba(255,255,255,0.75)"),
        updatemenus=[dict(type="buttons", direction="right", buttons=botones,
                          x=0, y=1.15, xanchor="left", yanchor="top",
                          bgcolor="#f8fafc", showactive=True)],
        scene=escena(False))
    fig.show()

for nombre in modelos:
    grafica_3d(nombre)

### Interpretación de la visualización R3

- **PCA** muestra los documentos proyectados a tres dimensiones sin etiquetas.
- **Esfera** normaliza esas coordenadas; para dos vectores unitarios $u,v$, la relación es $\|u-v\|_2^2=2-2\cos(u,v)$.
- **Clasificación** colorea las predicciones de los modelos supervisados originales; los colores **no** provienen de un nuevo entrenamiento 3D.
- **Aciertos/errores** contrasta las predicciones con las etiquetas verdaderas.
- **Vecinos 3D** y **Vecinos originales** muestran cómo cambia la selección de documentos cercanos a la noticia al proyectar y normalizar.
- **Categorías reales** permite examinar la estructura temática conocida.

La proyección PCA conserva solo una fracción de la información inicial. Las distancias y vecinos sobre la esfera 3D pueden diferir de los calculados en miles de dimensiones. **Las conclusiones de clasificación deben basarse en la evaluación del conjunto de prueba, no en la apariencia de la esfera.**
